# Dataset Insights

이 노트북은 BigQuery 데이터셋 레벨에 대해 Gemini 기반의 설명 작성을 지원하는 'DATA_DOCUMENTATION' 타입의 Knowledge Catalog DataScan 리소스를 생성하고 실행하여, 데이터셋 설명을 자동으로 생성하고 동기화하는 파이프라인입니다.

### 학습 목표
1. **데이터셋 레벨 Data Insights**: 테이블 단위가 아닌 BigQuery 데이터셋 단위로 전체를 요약 설명하는 데이터 문서화(Data Documentation) 스캔 아키텍처를 이해합니다.
2. **REST API 통합 연동**: 직접 HTTP 요청을 설계하여 Knowledge Catalog의 DataScan 리소스 및 개별 비동기 작업(Job) 상태를 기동하고 조회하는 로직을 구동합니다.
3. **데이터셋 메타데이터 갱신**: 생성 결과를 획득하여 BigQuery 데이터셋 설명 필드를 업데이트하고 퍼블리싱 라벨을 구성합니다.

### 작업 파이프라인

1. **환경 초기화**: BigQuery 클라이언트를 초기화하고 REST 호출을 위한 OAuth2 토큰을 발급받습니다.
2. **API 호출 공통 함수 정의**: OAuth2 Bearer 토큰 인증 헤더를 전송하여 Knowledge Catalog 및 DataScan REST API를 호출하는 헬퍼 함수를 정의합니다.
3. **데이터셋 DataScan 관리 및 모니터링**: 대상 데이터셋에 대해 Gemini 기반의 설명 작성을 지원하는 Knowledge Catalog DataScan 리소스를 생성하고, 완료될 때까지 상태를 폴링하여 대기합니다.
4. **메타데이터 추출 및 BigQuery 동기화**: 생성된 데이터셋 한글 설명 메타데이터를 BigQuery 데이터셋 설명으로 복사 동기화하고 공식 시스템 라벨을 부착합니다.

## Step 1: 환경 초기화

In [ ]:
import os
import json
import time
import urllib.request
import urllib.error
import ssl
import google.auth
from google.auth.transport.requests import Request, AuthorizedSession
from google.cloud import bigquery
import re

# 1. GCP 프로젝트 및 리전 정보 정의
credentials, PROJECT_ID = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
DATASET_ID = "thelook_ecommerce"
LOCATION = "us-central1"

# 2. REST API 호출을 위해 공식 AuthorizedSession 초기화
authed_session = AuthorizedSession(credentials)

# BigQuery API 호출용 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print(f"Project: {PROJECT_ID}, Dataset: {DATASET_ID}, 리전: {LOCATION}")
print("인프라 및 인증 설정 완료!")

## Step 2: API 호출 공통 함수 정의

In [ ]:
# [공통 유틸리티] Knowledge Catalog REST API 전송 함수
def make_rest_request(url, method="GET", body_dict=None, max_retries=5):
    """
    AuthorizedSession을 사용하여 OAuth2 토큰 관리 및 필요한 헤더(x-goog-user-project 등)를 자동으로 처리하여 REST API 통신을 수행합니다.
    429 Quota Exceeded 에러 발생 시 지수 백오프 기반으로 재시도합니다.
    """
    retries = 0
    backoff = 2  # 시작 대기 시간 (초)
    
    while True:
        try:
            if method == "GET":
                response = authed_session.get(url, timeout=60)
            elif method == "POST":
                response = authed_session.post(url, json=body_dict, timeout=60)
            elif method == "DELETE":
                response = authed_session.delete(url, timeout=60)
            else:
                response = authed_session.request(method, url, json=body_dict, timeout=60)
                
            # API 호출 중 에러가 발생한 경우 상세 에러 응답 바디를 파싱하여 예외를 발생시킵니다.
            if response.status_code >= 400:
                # 429 Quota/Rate Limit 에러 시 지수 백오프 기반 재시도 처리
                if response.status_code == 429 and retries < max_retries:
                    print(f"  [429 Quota Exceeded] {backoff}초 후 재시도 합니다 (재시도: {retries + 1}/{max_retries})...")
                    time.sleep(backoff)
                    retries += 1
                    backoff *= 2
                    continue
                raise Exception(f"HTTP Error {response.status_code} - {response.text}")
                
            return response.json()
            
        except Exception as e:
            if "HTTP Error" in str(e):
                raise e
            # 다른 네트워크 오류인 경우 재시도 처리
            if retries < max_retries:
                time.sleep(backoff)
                retries += 1
                backoff *= 2
                continue
            raise e

## Step 3: 데이터셋 DataScan 관리 및 메타데이터 추출

In [ ]:
def get_or_create_dataset_datascan(dataset_id):
    """
    대상 데이터셋에 대해 Gemini 기반의 설명 작성을 지원하는 'DATA_DOCUMENTATION' 타입의 Knowledge Catalog DataScan 리소스를 조회하고, 없으면 새로 생성합니다.
    """
    scan_id = f"ds-thelook-ds-{dataset_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Dataset DataScan 리소스 로드 성공: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e):
            print(f"  -> 새로운 Dataset DataScan 생성 중: {scan_id}...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            body = {
                "data": {
                    # Dataplex DataScan이 빅쿼리 데이터셋을 탐색할 수 있도록 리전 경로를 명시합니다.
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/locations/{LOCATION}/datasets/{dataset_id}"
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_DOCUMENTATION",
                "dataDocumentationSpec": {
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            # 리소스 생성이 완료될 때까지 LRO 상태 폴링 대기
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Dataset DataScan 생성 실패: {op_status['error']}")
                    break
                time.sleep(2)
            print(f"  -> Dataset DataScan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

def run_datascan_and_wait(scan_id):
    """
    Knowledge Catalog DataScan의 실행 Job을 기동하고, 완료될 때까지 주기적으로 상태를 조회(폴링)하여 대기합니다.
    """
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중...")
    run_res = make_rest_request(run_url, method="POST")
    
    job_name = run_res["job"]["name"]
    job_id = run_res["job"]["uid"]
    print(f"  -> 실행 Job ID: {job_id} (완료 대기 시작...)")
    
    # ?view=FULL 파라미터를 추가하여 상세 분석 결과 정보까지 포함해 조회
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}?view=FULL"
    while True:
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 상태: {state}")
        
        if state == "SUCCEEDED":
            print("  -> 설명 생성 성공!")
            break
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 오류 발생: {state}")
            
        time.sleep(10)

def fetch_dataset_generated_description(dataset_id):
    """
    Knowledge Catalog Entry API를 호출하여 생성된 데이터셋 설명(Descriptions)의 세부 속성(Aspect) 데이터 본문을 조회합니다.
    """
    entry_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_id}?view=ALL"
    entry_data = make_rest_request(entry_url, method="GET")
    
    aspects = entry_data.get("aspects", {})
    desc_key = [k for k in aspects.keys() if "descriptions" in k]
    if not desc_key:
        print(f"  -> [{dataset_id}] 생성된 설명(Descriptions) Aspect가 존재하지 않습니다.")
        return None
        
    return aspects[desc_key[0]]["data"]

def apply_and_publish_dataset_description(dataset_id, desc_data):
    """
    Knowledge Catalog에서 생성된 한글 설명 메타데이터를 BigQuery 데이터셋 설명으로 복사 동기화하고,
    스캔 결과를 연동하기 위한 공식 시스템 라벨을 데이터셋에 부착합니다.
    """
    if not desc_data:
        return
        
    dataset_ref = bq_client.dataset(dataset_id)
    dataset = bq_client.get_dataset(dataset_ref)
    
    # 1. 데이터셋 설명 업데이트
    dataset.description = desc_data.get("description", dataset.description)
    
    # 2. 공식 데이터 설명 퍼블리싱 라벨 추가 (데이터셋 라벨)
    labels = dict(dataset.labels or {})
    scan_id = f"ds-thelook-ds-{dataset_id}".lower().replace("_", "-")
    labels["dataplex-data-documentation-published-scan"] = scan_id
    labels["dataplex-data-documentation-published-project"] = PROJECT_ID
    labels["dataplex-data-documentation-published-location"] = LOCATION
    dataset.labels = labels
    
    # 3. BigQuery에 설명 및 라벨 업데이트 전송
    bq_client.update_dataset(dataset, ["description", "labels"])
    print(f"  [SUCCESS] {dataset_id} 데이터셋 설명 주입 완료!")

## Step 4: 메타데이터 추출 및 BigQuery 동기화

In [ ]:
# 메인 실행 파이프라인
print(f"=== 데이터셋 설명 자동화 시작: {DATASET_ID} ===")

# 0. 언어 지침 주입 (Language Directive)
# Gemini 모델이 데이터셋 설명을 한국어로 작성하도록 데이터셋의 기존 설명에 지침을 미리 삽입합니다.
dataset_ref = bq_client.dataset(DATASET_ID)
dataset = bq_client.get_dataset(dataset_ref)
dataset.description = "Generate dataset descriptions using the Korean language"
bq_client.update_dataset(dataset, ["description"])
print(f"  -> {DATASET_ID} 데이터셋에 한국어 작성 지침 주입 완료.")

# 1. Dataset DataScan 생성 또는 로드
scan_id = get_or_create_dataset_datascan(DATASET_ID)

# 2. DataScan 실행 및 완료 대기
run_datascan_and_wait(scan_id)

# 3. 생성된 설명 메타데이터 조회
desc_data = fetch_dataset_generated_description(DATASET_ID)

if desc_data:
    print(f"\n[생성된 데이터셋 설명 요약]")
    print(f" - 설명: {desc_data.get('description')}")
    
    # 4. BigQuery 데이터셋에 설명 및 라벨 반영
    apply_and_publish_dataset_description(DATASET_ID, desc_data)
else:
    print("설명 데이터를 가져오지 못했습니다.")

print(f"\n=== 데이터셋 설명 자동화 완료: {DATASET_ID} ===")